# Static XAS Energy Section Check

This notebook validates the stored `/nominal_energies` section axis against the undulator `set_wavelength_1` saved in the processed static-XAS H5.

For wavelength-derived processing, `/nominal_energies` should match `1239.841984 / set_wavelength_1_nm` section by section. If OPIS photon energy is saved in a future processed file, the helper below will report that as an additional diagnostic.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path

mpl_cache = Path(tempfile.gettempdir()) / "glycine26-matplotlib-cache"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 160})


def find_repo_root(start: Path) -> Path:
    for parent in (start.resolve(), *start.resolve().parents):
        if (parent / "analysis" / "scripts" / "compute_static_xas.py").exists():
            return parent
    raise RuntimeError("Could not find repo root from notebook path.")


REPO_ROOT = find_repo_root(Path.cwd())
SCRIPTS_DIR = REPO_ROOT / "analysis" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

import config as path_config

RUN_NO = 58794
H5_PATH = Path(path_config.COMBINED_DIR).parent / "xas_static" / f"run{RUN_NO}_static_xas.h5"

HC_EV_NM = 1239.8419843320026
EXPECTED_STEP_EV = -0.5
STEP_TOL_EV = 0.08
DUPLICATE_TOL_EV = 0.08
CONSTANT_OFFSET_TOL_EV = 0.15

H5_PATH


## Read per-section setpoint summary

In [ ]:
def decode_h5_strings(values) -> list[str]:
    return [v.decode("utf-8") if isinstance(v, bytes) else str(v) for v in values]


def feature_index(feature_names: list[str], name: str) -> int | None:
    try:
        return feature_names.index(name)
    except ValueError:
        return None


def finite_positive(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    return values[np.isfinite(values) & (values > 0)]


def scalar_stats(values: np.ndarray) -> dict[str, float | int]:
    values = finite_positive(values)
    if values.size == 0:
        return {"n_finite": 0, "p1": np.nan, "p50": np.nan, "p99": np.nan, "n_unique": 0}
    return {
        "n_finite": int(values.size),
        "p1": float(np.nanpercentile(values, 1)),
        "p50": float(np.nanmedian(values)),
        "p99": float(np.nanpercentile(values, 99)),
        "n_unique": int(np.unique(np.round(values, 8)).size),
    }


def load_energy_section_summary(h5_path: Path) -> tuple[pd.DataFrame, list[str]]:
    if not h5_path.exists():
        raise FileNotFoundError(h5_path)

    rows = []
    with h5py.File(h5_path, "r") as h5:
        required = ["n_shots", "nominal_energies", "ml/features", "ml/feature_names"]
        missing = [name for name in required if name not in h5]
        if missing:
            raise KeyError(f"Missing required datasets in {h5_path}: {missing}")

        n_shots = h5["n_shots"][...].astype(np.int64)
        nominal = h5["nominal_energies"][...].astype(np.float64)
        feature_names = decode_h5_strings(h5["ml/feature_names"][...])
        set_wl_idx = feature_index(feature_names, "set_wavelength_1")
        opis_e_idx = feature_index(feature_names, "opis_mean_photon_energy")
        opis_wl_idx = feature_index(feature_names, "opis_mean_wavelength")
        if set_wl_idx is None:
            raise KeyError("`set_wavelength_1` is not present in /ml/features. Add it to ML_FEATURE_NAMES and rerun compute_static_xas.py.")

        features = h5["ml/features"]
        for ie, energy in enumerate(nominal):
            n = int(n_shots[ie])
            row = {"section": ie, "nominal_eV": float(energy), "shots": n}

            wl_stats = scalar_stats(features[ie, :n, set_wl_idx])
            row.update({
                "set_wavelength_nm_p50": wl_stats["p50"],
                "set_wavelength_nm_p1": wl_stats["p1"],
                "set_wavelength_nm_p99": wl_stats["p99"],
                "set_wavelength_n_unique": wl_stats["n_unique"],
                "set_wavelength_n_finite": wl_stats["n_finite"],
            })
            if np.isfinite(wl_stats["p50"]):
                row["setpoint_eV"] = HC_EV_NM / float(wl_stats["p50"])
            else:
                row["setpoint_eV"] = np.nan

            if opis_e_idx is not None:
                opis_e = scalar_stats(features[ie, :n, opis_e_idx])
                row["opis_mean_photon_energy_eV_p50"] = opis_e["p50"]
                row["opis_mean_photon_energy_n_finite"] = opis_e["n_finite"]
            if opis_wl_idx is not None:
                opis_wl = scalar_stats(features[ie, :n, opis_wl_idx])
                row["opis_mean_wavelength_nm_p50"] = opis_wl["p50"]
                row["opis_mean_wavelength_n_finite"] = opis_wl["n_finite"]

            rows.append(row)

    df = pd.DataFrame(rows)
    df["setpoint_minus_nominal_eV"] = df["setpoint_eV"] - df["nominal_eV"]
    df["nominal_step_eV"] = df["nominal_eV"].diff()
    df["setpoint_step_eV"] = df["setpoint_eV"].diff()
    df["duplicate_previous_setpoint"] = df["setpoint_step_eV"].abs() <= DUPLICATE_TOL_EV
    df.loc[df.index == 0, "duplicate_previous_setpoint"] = False
    df["step_matches_expected"] = (df["setpoint_step_eV"] - EXPECTED_STEP_EV).abs() <= STEP_TOL_EV
    df.loc[df.index == 0, "step_matches_expected"] = True
    return df, feature_names


In [ ]:
summary, feature_names = load_energy_section_summary(H5_PATH)
print(f"processed H5: {H5_PATH}")
print("available wavelength/energy diagnostics:", [name for name in feature_names if "wavelength" in name or "opis" in name or "energy" in name])
display_cols = [
    "section", "nominal_eV", "shots", "set_wavelength_nm_p50", "setpoint_eV",
    "setpoint_minus_nominal_eV", "setpoint_step_eV", "duplicate_previous_setpoint",
    "step_matches_expected", "set_wavelength_n_unique",
]
display(summary[display_cols].style.format({
    "nominal_eV": "{:.3f}",
    "set_wavelength_nm_p50": "{:.6f}",
    "setpoint_eV": "{:.3f}",
    "setpoint_minus_nominal_eV": "{:+.3f}",
    "setpoint_step_eV": "{:+.3f}",
}))


## Automated sanity checks

In [ ]:
def run_section_checks(df: pd.DataFrame) -> pd.DataFrame:
    checks = []

    finite_setpoint = np.isfinite(df["setpoint_eV"].to_numpy())
    checks.append({
        "check": "finite set_wavelength_1 setpoint for every section",
        "passed": bool(np.all(finite_setpoint)),
        "detail": f"{int(finite_setpoint.sum())}/{len(df)} sections finite",
    })

    bad_step = df.index[(~df["step_matches_expected"]) & (df.index > 0)].to_numpy()
    checks.append({
        "check": f"adjacent setpoint step is {EXPECTED_STEP_EV:+.2f} +/- {STEP_TOL_EV:.2f} eV",
        "passed": bool(bad_step.size == 0),
        "detail": "bad sections: " + (", ".join(map(str, bad_step.tolist())) if bad_step.size else "none"),
    })

    duplicate_sections = df.index[df["duplicate_previous_setpoint"]].to_numpy()
    checks.append({
        "check": "no adjacent sections share the same setpoint",
        "passed": bool(duplicate_sections.size == 0),
        "detail": "duplicates after previous section: " + (", ".join(map(str, duplicate_sections.tolist())) if duplicate_sections.size else "none"),
    })

    delta = df.loc[finite_setpoint, "setpoint_minus_nominal_eV"].to_numpy(dtype=float)
    if delta.size:
        median_delta = float(np.nanmedian(delta))
        resid = np.abs(delta - median_delta)
        max_resid = float(np.nanmax(resid))
    else:
        median_delta = np.nan
        max_resid = np.nan
    checks.append({
        "check": f"setpoint-minus-nominal offset is constant within {CONSTANT_OFFSET_TOL_EV:.2f} eV",
        "passed": bool(np.isfinite(max_resid) and max_resid <= CONSTANT_OFFSET_TOL_EV),
        "detail": f"median offset={median_delta:+.3f} eV, max residual={max_resid:.3f} eV",
    })

    return pd.DataFrame(checks)


checks = run_section_checks(summary)
display(checks)
if checks["passed"].all():
    print("PASS: section order is consistent with the saved wavelength setpoint.")
else:
    print("WARNING: section assignment needs inspection before using nominal_energies as the XAS axis.")


## Diagnostic plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8), constrained_layout=True)

axes[0].plot(summary["section"], summary["nominal_eV"], "o-", label="stored /nominal_energies")
axes[0].plot(summary["section"], summary["setpoint_eV"], "s-", label="set wavelength -> eV")
axes[0].set_xlabel("section index")
axes[0].set_ylabel("energy (eV)")
axes[0].legend()
axes[0].grid(alpha=0.25)

axes[1].plot(summary["section"], summary["setpoint_step_eV"], "o-")
axes[1].axhline(EXPECTED_STEP_EV, color="black", lw=1, ls="--", label="expected step")
axes[1].axhspan(EXPECTED_STEP_EV - STEP_TOL_EV, EXPECTED_STEP_EV + STEP_TOL_EV, color="0.85", zorder=-1)
axes[1].set_xlabel("section index")
axes[1].set_ylabel("setpoint step (eV)")
axes[1].legend()
axes[1].grid(alpha=0.25)

axes[2].plot(summary["section"], summary["setpoint_minus_nominal_eV"], "o-")
axes[2].axhline(np.nanmedian(summary["setpoint_minus_nominal_eV"]), color="black", lw=1, ls="--", label="median")
axes[2].set_xlabel("section index")
axes[2].set_ylabel("setpoint - nominal (eV)")
axes[2].legend()
axes[2].grid(alpha=0.25)

plt.show()


## Duplicate setpoint groups

In [ ]:
grouped = (
    summary.assign(setpoint_eV_round=summary["setpoint_eV"].round(2))
    .groupby("setpoint_eV_round", as_index=False)
    .agg(
        n_sections=("section", "size"),
        sections=("section", lambda x: ", ".join(map(str, x))),
        nominal_sections=("nominal_eV", lambda x: ", ".join(f"{v:.1f}" for v in x)),
        total_shots=("shots", "sum"),
    )
    .sort_values("setpoint_eV_round", ascending=False)
)
duplicate_groups = grouped[grouped["n_sections"] > 1]
if duplicate_groups.empty:
    print("No duplicated setpoint groups at 0.01 eV rounding.")
else:
    print("Duplicated setpoint groups: these are likely split shutter sections at one energy setpoint.")
    display(duplicate_groups)

display(grouped)


Interpretation rule of thumb:

- For wavelength-derived processing, `setpoint_eV` and stored `nominal_eV` should agree up to rounding.
- A zero setpoint step between adjacent sections means the shutter sectioning split one physical energy step into multiple open blocks.
- If duplicated setpoints exist, combine those sections during downstream per-energy aggregation.